<a href="https://colab.research.google.com/github/deji4things2000/mlpro/blob/master/Decoder_Only_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Decoder-Only Transformer

In [ ]:
import random
import numpy as np
import torch

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------------
# 1. Load text file HERE
# --------------------
# Put your file "shakespeare.txt" in the same directory as this script / notebook.
with open("/content/drive/MyDrive/shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# --------------------
# 2. Build character vocabulary
# --------------------
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):  # string -> list[int]
    return [stoi[c] for c in s]

def decode(ids):  # list[int] -> string
    return "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)

# train/val split
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

# --------------------
# 3. Hyperparameters
# --------------------
device     = "cuda" if torch.cuda.is_available() else "cpu"
block_size = 128     # context length
batch_size = 32
d_model    = 128
n_heads    = 4
n_layers   = 2
learning_rate = 3e-4
max_iters  = 3000
eval_interval = 500

# --------------------
# 4. Data loader
# --------------------
def get_batch(split):
    src = train_data if split == "train" else val_data
    ix = torch.randint(0, len(src) - block_size - 1, (batch_size,))
    x = torch.stack([src[i:i+block_size]     for i in ix])   # (B, T)
    y = torch.stack([src[i+1:i+block_size+1] for i in ix])   # (B, T)
    return x.to(device), y.to(device)

# --------------------
# 5. Model components
# --------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_head = d_model // n_heads
        self.n_heads = n_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape

        q = self.Wq(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.Wk(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.Wv(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        # scaled dot-product attention
        scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)

        # causal mask (lower-triangular)
        mask = torch.tril(torch.ones(T, T, device=x.device))
        scores = scores.masked_fill(mask == 0, float("-inf"))

        attn = F.softmax(scores, dim=-1)
        out = attn @ v                              # (B, n_heads, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, D)  # (B, T, D)
        return self.Wo(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x):
        # Pre-LN residual block
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class CharTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb   = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        assert T <= self.block_size, "Sequence length exceeds block_size"

        pos = torch.arange(T, device=idx.device).unsqueeze(0)  # (1, T)
        x = self.token_emb(idx) + self.pos_emb(pos)            # (B, T, D)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)                                  # (B, T, vocab_size)
        return logits

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits = self(idx_cond)            # (B, T, vocab_size)
            logits = logits[:, -1, :]          # last time step
            probs = F.softmax(logits, dim=-1)  # (B, vocab_size)
            next_id = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

# --------------------
# 6. Instantiate model
# --------------------
model = CharTransformer(vocab_size, d_model, n_heads, n_layers, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# --------------------
# 7. Training loop
# --------------------
for step in range(max_iters):
    model.train()
    xb, yb = get_batch("train")        # (B, T)

    logits = model(xb)                 # (B, T, vocab_size)
    B, T, V = logits.shape
    loss = criterion(
        logits.view(B * T, V),
        yb.view(B * T)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0:
        model.eval()
        with torch.no_grad():
            xval, yval = get_batch("val")
            logits_val = model(xval)
            Bv, Tv, Vv = logits_val.shape
            val_loss = criterion(
                logits_val.view(Bv * Tv, Vv),
                yval.view(Bv * Tv)
            )
        print(f"step {step}: train loss {loss.item():.4f}, val loss {val_loss.item():.4f}")

# --------------------
# 8. Generate some text
# --------------------
model.eval()
start = torch.zeros((1, 1), dtype=torch.long, device=device)  # start with "<first char index 0>"
sample_ids = model.generate(start, max_new_tokens=500)[0].tolist()
print(decode(sample_ids))


step 0: train loss 4.3297, val loss 4.2365
step 500: train loss 2.4425, val loss 2.4273
step 1000: train loss 2.2109, val loss 2.2469
step 1500: train loss 2.0065, val loss 2.0517
step 2000: train loss 1.9127, val loss 1.9588
step 2500: train loss 1.7772, val loss 2.0133

KING HING arreate, soul, Son Must,
As stalake lake. forly exity,
As thell that eathy dess: with letter trueling a you sheeles:
Ay, theirm ones thy gatere.

PORTUS:
That is oll uniding yets bliesed that
Have thoplen thesor
Thou night flecond,'s too should grave may sidick
And stet partisits he dumb lay!

AUTOLUS:
No, an, You by are the his lead you diere
knighteld, beary this do comented his bet a to tore
From? came sonset be that cantash
Tnewsicess mouct?

wi'st i' came exreath bey his
to rived 


Sampling Model

In [2]:
# ------- after training loop -------

prompt = "JULIET:"  # any short string prompt
prompt_ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

model.eval()
with torch.no_grad():
    generated_ids = model.generate(prompt_ids, max_new_tokens=400)[0].tolist()

generated_text = decode(generated_ids)
print("===== SAMPLE =====")
print(generated_text)


===== SAMPLE =====
JULIET:
What, dears joyence make!
On, there'd slawdedated so, are ding him this pluts
Their squarme of mare you.

CORDONUCESBOLIESS IOS Crovabed:
So, gonry: it. though onet foreal and her the stribe?

Firlown!

MINABET:
Which buepsir, Clance, ster!, thy Richead.

FFRIV:
Gearail, wongs mark'

WARWARC IV:
For not way of quarmagher my stame.
So minkES:
Bard.

HARD OF ANGARE:

DUKE VINCETER:
These dright a a
